# Exponential Search

Objetivo: Localizar a faixa que contem o alvo dobrando o indice a cada passo e, em seguida, aplicar binary search nessa faixa. Ideal quando o tamanho total da colecao e desconhecido ou muito grande.

Complexidade:
- Tempo: O(log n)
- Espaco: O(1) iterativo | O(log n) se o binary search for recursivo

Categoria: Busca | Tecnica: Expansao exponencial + Divisao e Conquista

## Diagrama do fluxo

```mermaid
flowchart TD
    A[Inicio] --> B{arr 0 == target ?}
    B -->|Sim| C[Retornar 0]
    B -->|Nao| D[i = 1]
    D --> E{i < n AND arr i <= target ?}
    E -->|Sim, continuar dobrando| F[i = i * 2]
    F --> E
    E -->|Nao, faixa encontrada| G[Binary search em  arr i//2 .. min i n-1]
    G --> H{Encontrou ?}
    H -->|Sim| I[Retornar indice]
    H -->|Nao| J[Retornar -1]
```

## Fundamentos

Exponential search resolve um problema que binary search nao consegue tratar diretamente: **o tamanho da colecao e desconhecido**. Em streams de dados, sequencias potencialmente infinitas ou APIs paginadas onde nao sabemos o total de registros, binary search nao pode ser iniciado porque nao existe um `high` definido.

A estrategia e dupla:
1. **Fase exponencial**: dobrar o indice (1, 2, 4, 8, 16, ...) ate ultrapassar o alvo. Essa fase encontra a faixa em O(log n) comparacoes.
2. **Fase binaria**: aplicar binary search na faixa `[i//2, min(i, n-1)]`. O tamanho dessa faixa e no maximo `i//2`, e como `i ≈ 2 * posicao_real`, o binary search tambem custa O(log n).

O algoritmo e especialmente eficiente quando o alvo esta **proximo do inicio da colecao**: enquanto binary search sempre começa do meio, exponential search chega ate a posicao `p` em apenas log2(p) passos.

Casos de uso reais: busca em logs de audit com bilhoes de registros, recuperacao de eventos em streams Kafka sem offset inicial, busca em colecoes lazy (geradores Python, paginacao de APIs REST).

In [ ]:
def binary_search_range(arr, target, low, high):
    """
    Iterative binary search within [low, high].
    Used internally by exponential search.
    Time: O(log(high - low))
    """
    while low <= high:
        mid = low + (high - low) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1


def exponential_search(arr, target):
    """
    Searches for target in a sorted array using exponential search.

    Phase 1 (exponential): find range [i//2, i] that may contain target.
    Phase 2 (binary):      binary search within that range.

    Args:
        arr: Sorted list of comparable elements.
        target: Value to search for.

    Returns:
        Index of target, or -1 if not found.

    Time:  O(log n)
    Space: O(1)
    """
    n = len(arr)
    if n == 0:
        return -1

    # Edge case: first element
    if arr[0] == target:
        return 0

    # Phase 1: find range by doubling index
    i = 1
    while i < n and arr[i] <= target:
        i *= 2

    # Phase 2: binary search in [i//2, min(i, n-1)]
    return binary_search_range(arr, target, i // 2, min(i, n - 1))


# Validation
data = list(range(5, 505, 5))  # [5, 10, 15, ..., 500], n=100
print(f"n={len(data)}, range=[{data[0]}, {data[-1]}], step=5")
print()
print(f"exponential_search(arr, 5):    {exponential_search(data, 5)}")     # 0  (first)
print(f"exponential_search(arr, 50):   {exponential_search(data, 50)}")    # 9
print(f"exponential_search(arr, 255):  {exponential_search(data, 255)}")   # 50 (255 / 5 = 51o elemento, indice 50)
print(f"exponential_search(arr, 500):  {exponential_search(data, 500)}")   # 99 (last)

## Visualizacao passo a passo

In [ ]:
def exponential_search_verbose(arr, target):
    """
    Exponential search with detailed step trace.
    Returns (index_or_minus1, exp_steps, binary_steps).
    """
    n = len(arr)
    if n == 0:
        return -1, 0, 0

    print(f"Searching for {target!r} in array of n={n}")
    print("=" * 55)

    # Check first element
    if arr[0] == target:
        print(f"  arr[0] = {arr[0]} == target. Found immediately.")
        return 0, 1, 0

    # Phase 1: exponential doubling
    print("Phase 1: Exponential range finding")
    i = 1
    exp_steps = 0
    while i < n and arr[i] <= target:
        exp_steps += 1
        print(f"  [EXP {exp_steps}] i={i}: arr[{i}]={arr[i]} <= {target}, double to {i*2}")
        i *= 2

    low = i // 2
    high = min(i, n - 1)
    print(f"  [EXP stop] range locked: [{low}, {high}] (size={high - low + 1})")
    print()

    # Phase 2: binary search
    print("Phase 2: Binary search within range")
    binary_steps = 0
    l, h = low, high
    while l <= h:
        binary_steps += 1
        mid = l + (h - l) // 2
        if arr[mid] == target:
            print(f"  [BIN {binary_steps}] mid={mid}: arr[{mid}]={arr[mid]} == {target}  FOUND")
            print(f"\nResult: index {mid} | {exp_steps} exponential + {binary_steps} binary steps")
            return mid, exp_steps, binary_steps
        elif arr[mid] < target:
            print(f"  [BIN {binary_steps}] mid={mid}: arr[{mid}]={arr[mid]} < {target}, go right")
            l = mid + 1
        else:
            print(f"  [BIN {binary_steps}] mid={mid}: arr[{mid}]={arr[mid]} > {target}, go left")
            h = mid - 1

    print(f"\nResult: -1 | {exp_steps} exponential + {binary_steps} binary steps")
    return -1, exp_steps, binary_steps


arr = list(range(2, 130, 2))  # [2, 4, 6, ..., 128], n=64
exponential_search_verbose(arr, 44)

## Caso real: busca em stream de tamanho desconhecido

Em sistemas de processamento de eventos (Kafka, Kinesis, logs distribuidos), frequentemente precisamos localizar o primeiro evento apos um determinado timestamp sem conhecer o total de eventos existentes. Exponential search modela esse cenario naturalmente.

In [ ]:
from datetime import datetime, timedelta
from typing import Generator


class EventStream:
    """
    Simulates an ordered event stream where total size is unknown.
    Supports only forward indexed access (like Kafka offset reads).
    """

    def __init__(self, start: datetime, interval_seconds: int, count: int):
        self._start = start
        self._interval = timedelta(seconds=interval_seconds)
        self._count = count  # hidden from caller
        self._access_count = 0

    def get(self, offset: int) -> datetime | None:
        """Returns event timestamp at offset, or None if past end of stream."""
        self._access_count += 1
        if offset >= self._count:
            return None
        return self._start + self._interval * offset

    @property
    def access_count(self) -> int:
        return self._access_count


def exponential_search_stream(stream: EventStream, target: datetime) -> int:
    """
    Finds the offset of the first event with timestamp >= target in a stream
    of unknown size. Uses exponential range finding followed by binary search.

    Returns offset (int) or -1 if target is beyond stream end.
    Time: O(log p) where p is the position of target in the stream.
    """
    # Check first event
    first = stream.get(0)
    if first is None or first > target:
        return -1 if first is None else 0
    if first == target:
        return 0

    # Phase 1: find range by doubling offset
    i = 1
    while True:
        ev = stream.get(i)
        if ev is None or ev >= target:
            break
        i *= 2

    # Phase 2: binary search in [i//2, i]
    low, high = i // 2, i
    while low <= high:
        mid = low + (high - low) // 2
        ev_mid = stream.get(mid)
        if ev_mid is None:
            high = mid - 1
            continue
        if ev_mid == target:
            return mid
        elif ev_mid < target:
            low = mid + 1
        else:
            high = mid - 1

    return low  # insertion point (first event after target)


# Simulate a stream with 500,000 events at 1-second intervals
stream_start = datetime(2024, 1, 1, 0, 0, 0)
stream = EventStream(start=stream_start, interval_seconds=1, count=500_000)

# Search for an event ~5 hours in
target_ts = stream_start + timedelta(hours=5, minutes=37, seconds=22)

offset = exponential_search_stream(stream, target_ts)
found_ts = stream.get(offset)

print(f"Stream: 500,000 events starting {stream_start}, 1s interval")
print(f"Target timestamp: {target_ts}")
print(f"Found at offset:  {offset}")
print(f"Event timestamp:  {found_ts}")
print(f"Stream accesses:  {stream.access_count} (vs 500,000 for linear scan)")
print()

import math
expected_log = math.log2(offset) if offset > 0 else 0
print(f"Theoretical O(log {offset}) ≈ {expected_log:.1f} | actual accesses: {stream.access_count}")

## Analise de Complexidade

### Tempo

Seja `p` a posicao do alvo na colecao:

**Fase exponencial**: O(log p) iteracoes para dobrar de 1 ate 2^k >= p.

**Fase binaria**: A faixa `[i//2, i]` tem tamanho `i//2 <= p`. Binary search nessa faixa custa O(log p).

**Total**: O(log p) + O(log p) = **O(log p)**.

Quando p << n (alvo proximo do inicio), exponential search e mais eficiente que binary search:
- Binary search: O(log n) -- começa sempre do meio de toda a colecao
- Exponential search: O(log p) -- concentra as comparacoes ao redor da posicao do alvo

### Espaco

**O(1)** com binary search iterativo. **O(log n)** se usar recursao.

### Comparacao de complexidade

| Algoritmo | Complexidade | Tamanho desconhecido | Eficiente para alvo no inicio |
|---|---|---|---|
| Linear Search | O(n) | Sim | Nao |
| Binary Search | O(log n) | Nao | Nao |
| Jump Search | O(sqrt n) | Nao | Parcialmente |
| Exponential Search | O(log p) | Sim | Sim |

In [ ]:
import timeit
import random


def binary_search_bench(arr, target):
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = low + (high - low) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1


N = 1_000_000
data = list(range(N))
RUNS = 500

scenarios = [
    ("Target at position 10   (near start)",  10),
    ("Target at position 1000 (near start)",  1000),
    ("Target at position n/2  (middle)    ",  N // 2),
    ("Target at position n-1  (end)        ", N - 1),
]

print(f"n={N:,}")
print(f"{'Scenario':>40} | {'exponential (ms)':>18} | {'binary (ms)':>13}")
print("-" * 80)

for label, target in scenarios:
    t_exp = timeit.timeit(
        lambda d=data, t=target: exponential_search(d, t), number=RUNS
    ) / RUNS * 1000

    t_bin = timeit.timeit(
        lambda d=data, t=target: binary_search_bench(d, t), number=RUNS
    ) / RUNS * 1000

    print(f"{label:>40} | {t_exp:>18.4f} | {t_bin:>13.4f}")

print()
print("Nota: exponential search e mais rapido quando o alvo esta proximo do inicio.")
print("Para alvos no final, binary search e mais eficiente (começa do meio exato).")

## Exercicios

**Desafio 1:** Implemente `exponential_search_infinite(generator, target)` que busca um valor em uma sequencia ordenada **potencialmente infinita** representada por um gerador Python. O gerador produz valores sob demanda; seu codigo deve parar de consumi-lo assim que encontrar o alvo ou confirmar sua ausencia. Teste com um gerador de numeros primos.

**Desafio 2:** Compare exponential search com binary search para diferentes posicoes do alvo (1, 10, 100, 1000, 10000, n//2, n-1) em um array de n=1.000.000 elementos. Plote o numero de comparacoes de cada algoritmo em funcao da posicao do alvo para mostrar quando cada um e superior.

In [ ]:
from typing import Generator, TypeVar

T = TypeVar("T")


# Desafio 1: exponential search em gerador potencialmente infinito

def sieve_primes() -> Generator[int, None, None]:
    """Yields prime numbers indefinitely using a simple incremental sieve."""
    D: dict[int, int] = {}
    q = 2
    while True:
        if q not in D:
            yield q
            D[q * q] = q
        else:
            p = D.pop(q)
            x = q + p
            while x in D:
                x += p
            D[x] = p
        q += 1


def exponential_search_generator(gen: Generator[int, None, None], target: int):
    """
    Exponential search on a potentially infinite ordered generator.
    Buffers consumed values and performs binary search on the buffer.
    Stops consuming the generator as soon as the target range is found.

    Returns:
        (value_found, index_in_sequence, values_consumed_from_generator)
        or (-1, -1, values_consumed) if target is not in sequence.
    """
    buffer = []

    # Consume first element
    try:
        first = next(gen)
    except StopIteration:
        return -1, -1, 0

    buffer.append(first)
    if first == target:
        return first, 0, 1
    if first > target:
        return -1, -1, 1

    # Phase 1: double the buffer size until we overshoot
    i = 1
    while True:
        # Extend buffer to index i
        while len(buffer) <= i:
            try:
                buffer.append(next(gen))
            except StopIteration:
                # Generator exhausted before reaching i
                break

        last_idx = len(buffer) - 1
        if buffer[last_idx] >= target:
            break
        if len(buffer) <= i:
            # Generator exhausted and last value < target
            return -1, -1, len(buffer)
        i *= 2

    # Phase 2: binary search within buffer[i//2 .. min(i, last_idx)]
    low = i // 2
    high = min(i, len(buffer) - 1)
    while low <= high:
        mid = low + (high - low) // 2
        if buffer[mid] == target:
            return buffer[mid], mid, len(buffer)
        elif buffer[mid] < target:
            low = mid + 1
        else:
            high = mid - 1

    return -1, -1, len(buffer)


# Find specific prime numbers in the infinite prime sequence
targets = [2, 7, 97, 997, 9973]

print(f"{'Target':>8} | {'Found':>8} | {'Index in primes':>16} | {'Primes consumed':>16}")
print("-" * 58)

for t in targets:
    gen = sieve_primes()
    found, idx, consumed = exponential_search_generator(gen, t)
    print(f"{t:>8} | {found:>8} | {idx:>16} | {consumed:>16}")

print()

# Try a non-prime (should not be found)
gen = sieve_primes()
# Limit by stopping after consuming enough primes
# Search for 100 (not prime) -- will find -1 since 100 is not in prime sequence
# We'll use a bounded version to avoid infinite consumption
import itertools

def bounded_search_non_prime(target, limit):
    primes = list(itertools.islice(sieve_primes(), limit))
    idx = binary_search_range(primes, target, 0, len(primes) - 1)
    return idx

print(f"Searching for 100 (not prime) in first 200 primes: index={bounded_search_non_prime(100, 200)}")

In [ ]:
# Desafio 2: comparacao de comparacoes por posicao do alvo

def count_exp_comparisons(arr, target):
    """Returns number of comparisons made by exponential search."""
    n = len(arr)
    if n == 0:
        return 0
    comps = 1  # check first element
    if arr[0] == target:
        return comps
    i = 1
    while i < n and arr[i] <= target:
        comps += 1
        i *= 2
    low, high = i // 2, min(i, n - 1)
    l, h = low, high
    while l <= h:
        comps += 1
        mid = l + (h - l) // 2
        if arr[mid] == target:
            return comps
        elif arr[mid] < target:
            l = mid + 1
        else:
            h = mid - 1
    return comps


def count_bin_comparisons(arr, target):
    """Returns number of comparisons made by binary search."""
    low, high = 0, len(arr) - 1
    comps = 0
    while low <= high:
        comps += 1
        mid = low + (high - low) // 2
        if arr[mid] == target:
            return comps
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return comps


N = 1_000_000
arr = list(range(N))
positions = [1, 10, 100, 1_000, 10_000, N // 4, N // 2, N - 1]

print(f"n={N:,} | Comparisons per target position")
print(f"{'Position':>12} | {'Exponential':>13} | {'Binary':>8} | {'Winner':>12}")
print("-" * 52)

for pos in positions:
    target = arr[pos]
    exp_c = count_exp_comparisons(arr, target)
    bin_c = count_bin_comparisons(arr, target)
    winner = "exponential" if exp_c < bin_c else "binary" if bin_c < exp_c else "tie"
    print(f"{pos:>12,} | {exp_c:>13} | {bin_c:>8} | {winner:>12}")

## Referencias

1. Bentley, J. L., & Yao, A. C. "An almost optimal algorithm for unbounded searching". Information Processing Letters, 5(3), 82-87, 1976.
2. Cormen, T. H., et al. "Introduction to Algorithms" (3rd ed.). MIT Press, 2009, pp. 150-152.
3. Baeza-Yates, R., & Salinger, A. "Fast Intersection Algorithms for Sorted Sequences". Algorithms and Applications, Springer, 2010.